# Getting Started with NVFlare (TensorFlow)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVIDIA/NVFlare/blob/main/examples/getting_started/tf/nvflare_tf_getting_started.ipynb)

NVFlare is an open-source framework that allows researchers and
data scientists to seamlessly move their machine learning and deep
learning workflows into a federated paradigm.

## Basic Concepts
At the heart of NVFlare lies the concept of collaboration through
"tasks." An FL controller assigns tasks (e.g., training on local data) to one or more FL clients, processes returned
results (e.g., model weight updates), and may assign additional
tasks based on these results and other factors (e.g., a pre-configured
number of training rounds). The clients run executors which can listen for tasks and perform the necessary computations locally, such as model training. This task-based interaction repeats
until the experiment’s objectives are met. 

<img src="https://github.com/NVIDIA/NVFlare/blob/main/docs/resources/controller_executor_no_filter.png?raw=true" alt="NVIDIA FLARE Controller and Executor" width=75% height=75% />

## Setup environment

Install nvflare and dependencies:

In [ ]:
! pip install --ignore-installed blinker
! pip install nvflare tensorflow[and-cuda]

If running in Google Colab, download the source code for this example:

In [ ]:
! npx --yes degit -f NVIDIA/NVFlare/examples/advanced/cifar10/tf .

## Federated Averaging with NVFlare
Given the flexible controller and executor concepts, it is easy to implement different computing & communication patterns with NVFlare, such as [FedAvg](https://proceedings.mlr.press/v54/mcmahan17a?ref=https://githubhelp.com) and [cyclic weight transfer](https://academic.oup.com/jamia/article/25/8/945/4956468). 

The controller's `run()` routine is responsible for assigning tasks and processing task results from the Executors. 

### Server Code
In federated averaging, the server code is responsible for distributing the global model and aggregating model updates from clients. 

First, we provide a robust implementation of the [FedAvg](https://proceedings.mlr.press/v54/mcmahan17a?ref=https://githubhelp.com) algorithm with NVFlare. 

The server implements these main steps:
1. FL server initializes an initial model.
2. For each round (global iteration):
    - FL server samples available clients.
    - FL server sends the global model to clients and waits for their updates.
    - FL server aggregates all the `results` and produces a new global model.

In this example, we will directly use the default federated averaging algorithm provided by NVFlare utilizing the [FedAvgRecipe](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_opt.pt.recipes.fedavg.html#nvflare.app_opt.pt.recipes.fedavg.FedAvgRecipe) for PyTorch. 

There is no need to defined a customized server code for this example.

### Client Code 
Given a CIFAR10 [TensorFlow](https://www.tensorflow.org/) code example with a network defined at [src/tf_net.py](src/tf_net.py), we wish to adapt this centralized training code to something that can run in a federated setting.


On the client side, the training workflow is as follows:
1. Receive the model from the FL server.
2. Perform local training on the received global model
and/or evaluate the received global model for model
selection.
3. Send the new model back to the FL server.

Using NVFlare's client API, we can easily adapt machine learning code that was written for centralized training and apply it in a federated scenario.
For a general use case, there are three essential methods to achieve this using the Client API :
- `init()`: Initializes NVFlare Client API environment.
- `receive()`: Receives model from the FL server.
- `send()`: Sends the model to the FL server.

With these simple methods, the developers can use the Client API
to change their centralized training code to an FL scenario with
five lines of code changes as shown below.
```python
    import nvflare.client as flare
    
    flare.init() # 1. Initializes NVFlare Client API environment.
    input_model = flare.receive() # 2. Receives model from the FL server.
    for k, v in input_model.params.items():
        model.get_layer(k).set_weights(v) # 3. Loads model from NVFlare
    
    # original local training code
    model.fit(...)
    
    output_model = flare.FLModel(params={layer.name: layer.get_weights() for layer in model.layers}) # 4. Put the results in a new `FLModel`
    flare.send(output_model) # 5. Sends the model to the FL server.  
```

The full client training script is saved in a separate file, e.g. [./client.py](./client.py) doing CNN training on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset.

## Run an NVFlare Job
Now that we have our client training script that uses the Client API to receive models, run local training, and send results back to the FL server, we can put everything together using NVFlare's Recipe API.

#### 1. Define the initial model
First, we define the global model used to initialize the model on the FL server. See [model.py](model.py).

```python
from tensorflow.keras import layers, models

class TFNet(models.Sequential):
    def __init__(self, input_shape=(None, 32, 32, 3)):
        super().__init__()
        self._input_shape = input_shape
        # Do not specify input as we will use delayed built only during runtime of the model
        # self.add(layers.Input(shape=(32, 32, 3)))
        self.add(layers.Conv2D(32, (3, 3), activation="relu"))
        self.add(layers.MaxPooling2D((2, 2)))
        self.add(layers.Conv2D(64, (3, 3), activation="relu"))
        self.add(layers.MaxPooling2D((2, 2)))
        self.add(layers.Conv2D(64, (3, 3), activation="relu"))
        self.add(layers.Flatten())
        self.add(layers.Dense(64, activation="relu"))
        self.add(layers.Dense(10))
```

#### 2. Define a FedAvgRecipe
The `FedAvgRecipe` provides a simplified API for defining federated learning jobs with the FedAvg algorithm.

The recipe automatically configures:
- Server-side scatter-and-gather controller
- Weighted aggregator for combining client updates
- Client-side script runners
- Model persistence and selection components

We create a recipe with the job name, initial model, number of clients, training rounds, and the client training script.

In [ ]:
from model import TFNet

from nvflare.app_opt.tf.recipes.fedavg import FedAvgRecipe
from nvflare.recipe import SimEnv, add_experiment_tracking

n_clients = 2
num_rounds = 2

recipe = FedAvgRecipe(
    name="cifar10_tf_fedavg",
    min_clients=n_clients,
    num_rounds=num_rounds,
    initial_model=TFNet(),
    train_script="client.py",
)

#### 3. Add experiment tracking

In [ ]:
add_experiment_tracking(recipe, tracking_type="tensorboard")

#### 4. Run Job
Here, we run the job in a simulation environment.

In [ ]:
env = SimEnv(num_clients=n_clients)
run = recipe.execute(env)
print()
print("Job Status is:", run.get_status())
print("Result can be found in :", run.get_result())
print()

#### 5. Visualize the Training
You can use TensorBoard to show the experiment tracking curves by running

```bash
tensorboard --bind_all --logdir /tmp/nvflare/simulation/cifar10_tf_fedavg
```
in another terminal or directly show the training curves in the next notebook cell.

In [ ]:
%load_ext tensorboard
%tensorboard --bind_all --logdir /tmp/nvflare/simulation/cifar10_tf_fedavg

#### 6. Next steps

Continue with the steps described in the [README.md](README.md) to run more experiments with a more complex model and more advanced FL algorithms.